# Pretrain backbone — PTB-XL (supervised) + MIMIC-IV-ECG (self-supervised)

**Dự án ACS-ECG-AI (Vinmec).** Notebook 14 — chuẩn bị backbone đã pretrain để khởi tạo cho
fine-tune **STEMI** (notebook 15) và **OMI** (notebook 16) trên ACS-ECG 2026.

**Trạng thái: đã nối đầy đủ, chỉ cần đổi `RUN_MODE = "full"` và chạy toàn bộ notebook.**
Không còn placeholder — cụ thể:
- Đọc thật `ptbxl_database.csv` + `scp_statements.csv`, ánh xạ nhãn SCP-ECG → NORM/MI/OTHER_ABNORMAL.
- Đọc waveform thật bằng `wfdb.rdrecord()` cho cả PTB-XL và MIMIC-IV-ECG.
- Tiền xử lý khớp đúng `stemi_pipeline_optimized_v3.ipynb`: highpass **0,05Hz** (không phải 0,5Hz),
  bandpass Butterworth bậc 3, winsorize ±6mV.
- **Backbone là kiến trúc ResNet1D thật** lấy nguyên từ `stemi_pipeline_optimized_v3.ipynb` mục 11
  (bỏ head phân loại STEMI, giữ nguyên stem + residual blocks) — không phải kiến trúc giả lập.
- Cache tín hiệu dùng đúng pattern memmap resumable của pipeline STEMI (mục 6): 1 file `.npy` lớn
  mỗi nguồn dữ liệu, build local trước rồi backup định kỳ + cuối cùng lên Drive — không ghi hàng
  trăm nghìn file nhỏ trực tiếp lên Drive.
- MIMIC-IV-ECG: waveform mở, không cần credential PhysioNet (đã xác nhận qua notebook 09 mục 10).

**RUN_MODE=debug** chạy thật toàn bộ pipeline trên `DEBUG_LIMIT` bản ghi mỗi nguồn (mặc định 32) —
không dùng dữ liệu giả lập — để phát hiện lỗi tích hợp sớm trước khi chạy full.

**Nguyên tắc bắt buộc:**
- Đây là bước *pretrain*, không phải *external validation*.
- **KHÔNG** chỉnh sửa bất kỳ file nào trong `VinAMI_ACS/data/ptb-xl/1.0.3/v2_training_cache_v1`.
- TEST giữ riêng của ACS-ECG 2026 không được chạm ở notebook này.

In [ ]:
import os, sys, json, hashlib, random, time, re, shutil, subprocess, importlib.util, ast
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests

if importlib.util.find_spec("wfdb") is None:
    print("Đang cài wfdb ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
import wfdb

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except ImportError:
    raise SystemExit("Chưa có torch — chạy trên Colab GPU runtime (Runtime > Change runtime type > GPU).")

from google.colab import drive
drive.mount('/content/drive')

# ============================== CONFIG ==============================
RUN_MODE = "full"  # "debug" | "full" — đổi sang full khi đã sẵn sàng chạy thật, không cần sửa gì thêm

SEED = 42
FS = 500
SIGNAL_LEN = 5000
NUM_LEADS = 12
BP_LOW, BP_HIGH, BP_ORDER = 0.05, 40.0, 3
# Khớp đúng thông số hiện tại của stemi_pipeline_optimized_v3.ipynb (mục 2): highpass 0.05Hz
# theo chuẩn AHA/ACC/HRS cho ECG chẩn đoán (0.5Hz làm suy hao mức chênh ST tuyệt đối).
WINSORIZE_MV = 6.0

PREPROCESS_VERSION = "pretrain_v3_real"
DEBUG_LIMIT = 32   # số record MỖI nguồn khi RUN_MODE=debug -- pipeline chạy THẬT, không giả lập

DRIVE_ROOT = Path("/content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune")
DATA_DIR = DRIVE_ROOT / "data"
MIMIC_RAW_DIR = DATA_DIR / "mimic_raw"
MIMIC_MANIFEST_DIR = DATA_DIR / "mimic_manifest"
PTBXL_REF_DIR = DATA_DIR / "ptbxl_ref"
UNIFIED_CACHE_DIR = DATA_DIR / "unified_cache"        # Drive -- cache tín hiệu memmap (vài file lớn)
MODELS_DIR = DRIVE_ROOT / "models" / "backbone_checkpoints"
LOGS_DIR = DRIVE_ROOT / "logs"
MANIFESTS_DIR = DRIVE_ROOT / "manifests"

LOCAL_WORK_DIR = Path("/content/pretrain_work")       # local Colab -- scratch nhanh
LOCAL_CACHE_DIR = LOCAL_WORK_DIR / "cache"
MIMIC_LOCAL_WORK_DIR = LOCAL_WORK_DIR / "mimic_wfdb"   # Bước 2 tải waveform MIMIC vào đây

PTBXL_SOURCE_DIR = Path("/content/drive/MyDrive/VinAMI_ACS/data/ptb-xl/1.0.3")
PTBXL_FORBIDDEN_SUBDIR = PTBXL_SOURCE_DIR / "v2_training_cache_v1"  # CHỈ ĐỌC — không ghi vào đây

for d in [DATA_DIR, MIMIC_RAW_DIR, MIMIC_MANIFEST_DIR, PTBXL_REF_DIR, UNIFIED_CACHE_DIR,
          MODELS_DIR, LOGS_DIR, MANIFESTS_DIR, LOCAL_CACHE_DIR, MIMIC_LOCAL_WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"RUN_MODE={RUN_MODE} | device={DEVICE} | PREPROCESS_VERSION={PREPROCESS_VERSION}")
assert PTBXL_SOURCE_DIR.exists(), (
    f"Không tìm thấy PTB-XL tại {PTBXL_SOURCE_DIR} — kiểm tra shortcut VinAMI_ACS trong MyDrive"
)
assert (PTBXL_SOURCE_DIR / "ptbxl_database.csv").exists(), "Thiếu ptbxl_database.csv"
assert (PTBXL_SOURCE_DIR / "scp_statements.csv").exists(), "Thiếu scp_statements.csv"
print("wfdb", wfdb.__version__)

Đang cài wfdb ...
Mounted at /content/drive
RUN_MODE=full | device=cpu | PREPROCESS_VERSION=pretrain_v3_real
wfdb 4.3.1


## Bước 1 — Probe: kiểm tra dữ liệu đã có, lập kế hoạch subsample MIMIC

**Chỉ áp dụng khi `RUN_MODE="full"`.** MIMIC-IV-ECG đầy đủ ~800.000 bản ghi (~90–100GB tín hiệu
thô) — không tải toàn bộ. Ở `RUN_MODE="debug"`, notebook chỉ dùng `DEBUG_LIMIT` bản ghi nên bước
này không quan trọng (nhưng vẫn chạy để bạn quen luồng trước khi chuyển sang full).

In [ ]:
def bytes_per_record(num_leads=NUM_LEADS, signal_len=SIGNAL_LEN, dtype_bytes=2):
    return num_leads * signal_len * dtype_bytes

def human_gb(n_bytes):
    return n_bytes / (1024 ** 3)

N_MIMIC_TARGET = 100_000     # dùng khi RUN_MODE="full" — chỉnh theo dung lượng Drive còn trống
n_planned = DEBUG_LIMIT if RUN_MODE == "debug" else N_MIMIC_TARGET

est_bytes = n_planned * bytes_per_record()
print(f"Số record MIMIC sẽ xử lý ở RUN_MODE={RUN_MODE!r}: {n_planned:,}")
print(f"Dung lượng tín hiệu ước tính: ~{human_gb(est_bytes):.2f} GB (chưa gồm header/overhead)")
if RUN_MODE == "full":
    print("\n>>> Nếu số này quá lớn so với dung lượng Drive còn trống (Google Drive > Storage), "
          "chỉnh N_MIMIC_TARGET ở cell trên rồi chạy lại cell này trước khi sang Bước 2.")

Số record MIMIC sẽ xử lý ở RUN_MODE='full': 100,000
Dung lượng tín hiệu ước tính: ~11.18 GB (chưa gồm header/overhead)

>>> Nếu số này quá lớn so với dung lượng Drive còn trống (Google Drive > Storage), chỉnh N_MIMIC_TARGET ở cell trên rồi chạy lại cell này trước khi sang Bước 2.


## Bước 2 — Tải waveform MIMIC-IV-ECG (open access, không cần credential)

Không cần nhãn (self-supervised) nên chỉ cần danh sách bản ghi bất kỳ để random sample.
`record_list.csv` là file mục lục cơ bản của MIMIC-IV-ECG (khác `records_w_diag_icd10.csv` của
Ext-ICD, vốn cần credential). Việc tải waveform (.hea/.dat) từng bản ghi mở hoàn toàn, đã xác nhận
qua notebook 09 (mục 10). Chạy thật ở cả debug (n nhỏ) lẫn full — không giả lập.

In [ ]:
import zipfile

MIMIC_BASE = "https://physionet.org/files/mimic-iv-ecg/1.0/"
MIMIC_RECORD_LIST_URL = MIMIC_BASE + "record_list.csv"
MIMIC_RECORD_LIST_CACHE = MIMIC_MANIFEST_DIR / "record_list.csv"
WFDB_CACHE_ZIP = MIMIC_RAW_DIR / "mimic_wfdb_cache.zip"
CACHE_CHECKPOINT_EVERY = 10_000   # lưu cache lên Drive sau mỗi N record tải xong
MIMIC_SAMPLE_MANIFEST = MIMIC_MANIFEST_DIR / "mimic_selfsup_sample.csv"

def _quick_cache_complete(npy_p, meta_p, n_expected):
    """Bản rút gọn của _cache_complete() (định nghĩa đầy đủ ở Bước 4) -- dùng ở đây để kiểm tra
    SỚM xem cache đã xử lý HOÀN CHỈNH chưa, trước khi đụng đến bất kỳ file MIMIC thô nào."""
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= n_expected
    except (OSError, ValueError):
        return False

# --- Bỏ qua HOÀN TOÀN việc tải/giải nén MIMIC thô nếu cache đã xử lý xong từ trước ---
# Không chỉ tránh giải nén lại -- tránh CHẠM vào file MIMIC thô (zip lẫn file lẻ) hoàn toàn, vì Bước 4
# chỉ cần đúng 1 file .npy đã xử lý sẵn trên Drive, không cần đọc lại WFDB gốc nữa.
_MIMIC_SKIP_RAW = False
if MIMIC_SAMPLE_MANIFEST.exists():
    _saved_sample_df = pd.read_csv(MIMIC_SAMPLE_MANIFEST)
    _final_tag = f"mimic_{PREPROCESS_VERSION}_n{len(_saved_sample_df)}"
    _final_npy = UNIFIED_CACHE_DIR / f"{_final_tag}.npy"
    _final_meta = UNIFIED_CACHE_DIR / f"{_final_tag}.meta.json"
    if _quick_cache_complete(_final_npy, _final_meta, len(_saved_sample_df)):
        print(f"Tìm thấy cache MIMIC đã xử lý HOÀN CHỈNH từ lần chạy trước ({len(_saved_sample_df):,} "
              f"record, {_final_npy.name}) -- BỎ QUA HOÀN TOÀN tải/giải nén MIMIC thô. Bước 4 sẽ dùng "
              f"thẳng file cache đã có.")
        mimic_sample_df = _saved_sample_df
        _MIMIC_SKIP_RAW = True

def fetch_mimic_record_list(cache_path=MIMIC_RECORD_LIST_CACHE):
    if cache_path.exists():
        print(f"Dùng lại record_list.csv đã cache tại {cache_path}")
        return pd.read_csv(cache_path)
    print("Đang thử tải record_list.csv (open access, không auth)...")
    r = requests.get(MIMIC_RECORD_LIST_URL, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(
            f"Không tải được record_list.csv (HTTP {r.status_code}) -- file mục lục này có vẻ bị "
            f"gate, khác waveform từng bản ghi. Nhờ người có credential tải hộ MỘT file nhỏ này "
            f"(không cần cả Ext-ICD) rồi đặt vào {cache_path}."
        )
    cache_path.write_bytes(r.content)
    print(f"Tải xong record_list.csv ({len(r.content) / 1e6:.1f} MB)")
    return pd.read_csv(cache_path)

_REL_PATH_RE = re.compile(r"p\d{4}/p\d+/s\d+/\d+$")

def normalize_rel_path(fn: str) -> str:
    fn = str(fn).strip().lstrip("/")
    for ext in (".hea", ".dat"):
        if fn.endswith(ext):
            fn = fn[: -len(ext)]
    m = _REL_PATH_RE.search(fn)
    return m.group(0) if m else fn

def find_path_column(df):
    """Dò cột chứa path dạng pNNNN/... thay vì giả định cứng cột đầu tiên -- record_list.csv
    của MIMIC-IV-ECG có thể đặt tên cột path/file_name khác nhau tuỳ phiên bản."""
    for col in df.columns:
        sample = str(df[col].iloc[0])
        if _REL_PATH_RE.search(sample) or "/" in sample:
            return col
    return df.columns[0]

def sample_mimic_records(record_list_df, n_target, seed=SEED):
    rng = np.random.default_rng(seed)
    n = min(n_target, len(record_list_df))
    idx = rng.choice(len(record_list_df), size=n, replace=False)
    return record_list_df.iloc[idx].reset_index(drop=True)

def make_mimic_session(pool_maxsize=16):
    from requests.adapters import HTTPAdapter
    session = requests.Session()
    adapter = HTTPAdapter(pool_connections=pool_maxsize, pool_maxsize=pool_maxsize)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def download_one_record(rel_path, dest_dir=MIMIC_LOCAL_WORK_DIR, base_url=MIMIC_BASE, session=None):
    session = session or requests
    local_stub = dest_dir / Path(rel_path).name
    for ext in (".hea", ".dat"):
        dst = local_stub.with_suffix(ext)
        if dst.exists():
            continue
        r = session.get(base_url + rel_path + ext, timeout=60)
        if r.status_code != 200:
            return rel_path, f"HTTP {r.status_code}"
        dst.write_bytes(r.content)
    return rel_path, "ok"

LOCAL_WFDB_ZIP = MIMIC_LOCAL_WORK_DIR.parent / "mimic_wfdb_cache_local.zip"
_ZIPPED_FILES = set()  # theo dõi file nào đã nén, tránh nén lại toàn bộ mỗi checkpoint (bug đã gặp)

def save_wfdb_cache_to_drive(local_dir=MIMIC_LOCAL_WORK_DIR, cache_zip=WFDB_CACHE_ZIP,
                              local_zip=LOCAL_WFDB_ZIP):
    """CHỈ nén file MỚI (chưa có trong _ZIPPED_FILES) vào 1 file zip local bền vững trong suốt
    phiên, rồi copy zip đó lên Drive -- KHÔNG đọc/ghi lại toàn bộ file cũ mỗi lần checkpoint (bug
    cũ khiến mỗi checkpoint tốn hàng trăm giây, tăng dần theo tổng dữ liệu đã tải)."""
    all_files = sorted(local_dir.glob("*"))
    new_files = [f for f in all_files if f.name not in _ZIPPED_FILES]
    if not new_files:
        print("Không có file mới -- bỏ qua checkpoint.")
        return
    mode = "a" if local_zip.exists() else "w"
    with zipfile.ZipFile(local_zip, mode, zipfile.ZIP_STORED) as zf:
        for f in new_files:
            zf.write(f, arcname=f.name)
            _ZIPPED_FILES.add(f.name)
    shutil.copy2(local_zip, cache_zip)
    print(f"Đã lưu cache lên Drive: {cache_zip} ({cache_zip.stat().st_size / 1e6:.1f} MB) -- "
          f"thêm {len(new_files):,} file mới (không nén lại {len(_ZIPPED_FILES) - len(new_files):,} file cũ).")

def download_mimic_subset_open(paths, dest_dir=MIMIC_LOCAL_WORK_DIR, max_workers=8,
                                checkpoint_every=CACHE_CHECKPOINT_EVERY, max_fail_rate=0.2):
    """Vài record lẻ tẻ hỏng/thiếu trên PhysioNet là bình thường ở quy mô lớn -- chỉ raise nếu tỷ
    lệ lỗi vượt max_fail_rate (dấu hiệu access có vấn đề thật, không phải hỏng lẻ tẻ)."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t0 = time.time()
    results = {}
    session = make_mimic_session(pool_maxsize=max_workers * 2)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(download_one_record, p, dest_dir, MIMIC_BASE, session): p for p in paths}
        for i, fut in enumerate(as_completed(futs), 1):
            rel_path, status = fut.result()
            results[rel_path] = status
            if i % 200 == 0 or i == len(paths):
                print(f"  {i:>6}/{len(paths)}  ({time.time() - t0:.0f}s)")
            if checkpoint_every and i % checkpoint_every == 0:
                save_wfdb_cache_to_drive(dest_dir)
    n_fail = sum(1 for v in results.values() if v != "ok")
    fail_rate = n_fail / max(len(paths), 1)
    print(f"Xong. Lỗi: {n_fail}/{len(paths)} ({fail_rate:.1%})")
    if fail_rate > max_fail_rate:
        raise RuntimeError(
            f"Tỷ lệ lỗi quá cao ({fail_rate:.1%} > {max_fail_rate:.0%}) -- khả năng access có vấn đề "
            f"thật (không phải vài record hỏng lẻ tẻ). Kiểm tra kết nối/access rồi chạy lại cell này."
        )
    elif n_fail:
        print(f"  ({n_fail} record hỏng lẻ tẻ -- bình thường ở quy mô lớn, sẽ bị loại khỏi tập "
              f"dùng tiếp ở bước sau, không dừng cả notebook.)")
    return results

if not _MIMIC_SKIP_RAW:
    if WFDB_CACHE_ZIP.exists():
        print(f"Tìm thấy cache trên Drive ({WFDB_CACHE_ZIP.stat().st_size / 1e6:.1f} MB) -- giải nén...")
        with zipfile.ZipFile(WFDB_CACHE_ZIP) as zf:
            zf.extractall(MIMIC_LOCAL_WORK_DIR)
            _ZIPPED_FILES.update(zf.namelist())  # đánh dấu các file này ĐÃ có trong zip -- không nén lại
        shutil.copy2(WFDB_CACHE_ZIP, LOCAL_WFDB_ZIP)  # dùng luôn cache cũ làm zip local khởi điểm
        print(f"Đã giải nén {len(list(MIMIC_LOCAL_WORK_DIR.glob('*.hea'))):,} bản ghi từ cache "
              f"(đã đánh dấu {len(_ZIPPED_FILES):,} file trong zip cũ -- sẽ không nén lại).")

    MIN_ACCEPTABLE_EXISTING = 100  # nếu đã có sẵn ít nhất số này, không bắt buộc phải tải thêm được

    # Đếm record đã có sẵn từ cache trước khi cố tải thêm -- quan trọng để biết có thể "dùng tạm những
    # gì đã có" nếu PhysioNet đang chặn/rate-limit sau khi đã tải nhiều.
    _existing_hea = sorted(MIMIC_LOCAL_WORK_DIR.glob("*.hea"))
    n_existing = len(_existing_hea)
    print(f"Đã có sẵn {n_existing:,} record từ cache cục bộ.")

    n_to_fetch = DEBUG_LIMIT if RUN_MODE == "debug" else N_MIMIC_TARGET
    record_list_df = fetch_mimic_record_list()
    path_col = find_path_column(record_list_df)
    mimic_sample_df = sample_mimic_records(record_list_df, n_to_fetch)
    mimic_sample_df["_rel_path"] = mimic_sample_df[path_col].apply(normalize_rel_path)

    # Probe NHIỀU record thay vì 1 -- một vài record lẻ tẻ bị hỏng/thiếu trên PhysioNet là chuyện bình
    # thường ở dataset ~800k record, không có nghĩa là cả nguồn bị chặn. Chỉ coi là dấu hiệu access có
    # vấn đề nếu TẤT CẢ record thử đều fail.
    N_PROBE = min(5, len(mimic_sample_df))
    probe_results = []
    for _rel in mimic_sample_df["_rel_path"].iloc[:N_PROBE]:
        try:
            _r = requests.get(MIMIC_BASE + _rel + ".hea", timeout=30)
            _code = _r.status_code
        except requests.exceptions.RequestException as _e:
            # lỗi kết nối (timeout, refused, DNS...) -- không chỉ HTTP status khác 200 -- coi như fail
            _code = f"ERR:{type(_e).__name__}"
        probe_results.append((_rel, _code))
        print(f"  probe {_rel}.hea -> {_code}")

    n_probe_ok = sum(1 for _, code in probe_results if code == 200)
    print(f"Kiểm tra thử: {n_probe_ok}/{N_PROBE} OK")

    if n_probe_ok == 0 and n_existing >= MIN_ACCEPTABLE_EXISTING:
        # KHÔNG tải thêm được (có thể PhysioNet đang rate-limit tạm thời sau khi đã tải nhiều, hoặc
        # server trục trặc) -- nhưng đã có sẵn đủ record từ trước, KHÔNG cần chặn cả notebook lại.
        print(f"\n>>> Không tải thêm được lúc này (khả năng PhysioNet đang rate-limit tạm thời sau khi "
              f"đã tải {n_existing:,} record, hoặc server trục trặc) -- NHƯNG đã có sẵn {n_existing:,} "
              f"record từ cache, đủ để tiếp tục. Bỏ qua tải thêm ở lần chạy này, dùng luôn dữ liệu "
              f"hiện có. Muốn tải thêm sau, đợi 15-30 phút rồi chạy lại cell này.")
        mimic_sample_df = pd.DataFrame({"_rel_path": [p.stem for p in _existing_hea]})
    elif n_probe_ok == 0:
        raise RuntimeError(
            f"TẤT CẢ {N_PROBE} record thử đều lỗi VÀ chỉ có {n_existing:,} record sẵn có (dưới ngưỡng "
            f"{MIN_ACCEPTABLE_EXISTING:,}) -- không đủ để tiếp tục. Khả năng cao access bị chặn thật "
            f"(rate-limit sau khi tải nhiều, hoặc MIMIC-IV-ECG đổi chính sách truy cập). Thử lại sau "
            f"15-30 phút, hoặc giảm max_workers ở download_mimic_subset_open() để tải nhẹ nhàng hơn."
        )
    else:
        if n_probe_ok < N_PROBE:
            print(f"  (một vài record hỏng lẻ tẻ -- bình thường, {N_PROBE - n_probe_ok} record sẽ được "
                  f"ghi nhận lỗi riêng khi tải hàng loạt bên dưới, không dừng cả notebook.)")
        download_results = download_mimic_subset_open(mimic_sample_df["_rel_path"].tolist())
        _failed_paths = {p for p, status in download_results.items() if status != "ok"}
        if _failed_paths:
            mimic_sample_df = mimic_sample_df[~mimic_sample_df["_rel_path"].isin(_failed_paths)].reset_index(drop=True)
            print(f"Loại {len(_failed_paths):,} record tải lỗi khỏi tập dùng tiếp -- còn {len(mimic_sample_df):,} record.")
        save_wfdb_cache_to_drive()

    mimic_sample_df.to_csv(MIMIC_MANIFEST_DIR / "mimic_selfsup_sample.csv", index=False)
    print(f"MIMIC sẵn sàng: {len(mimic_sample_df):,} bản ghi.")

Tìm thấy cache trên Drive (10872.5 MB) -- giải nén...
Đã giải nén 89,976 bản ghi từ cache (đã đánh dấu 179,946 file trong zip cũ -- sẽ không nén lại).
Đã có sẵn 89,976 record từ cache cục bộ.
Dùng lại record_list.csv đã cache tại /content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune/data/mimic_manifest/record_list.csv
  probe p1794/p17947312/s48316038/48316038.hea -> 404
  probe p1535/p15357098/s47194783/47194783.hea -> 404
  probe p1968/p19687461/s45418062/45418062.hea -> 404
  probe p1363/p13631359/s49260715/49260715.hea -> 404
  probe p1500/p15002538/s42331345/42331345.hea -> 404
Kiểm tra thử: 0/5 OK

>>> Không tải thêm được lúc này (khả năng PhysioNet đang rate-limit tạm thời sau khi đã tải 89,976 record, hoặc server trục trặc) -- NHƯNG đã có sẵn 89,976 record từ cache, đủ để tiếp tục. Bỏ qua tải thêm ở lần chạy này, dùng luôn dữ liệu hiện có. Muốn tải thêm sau, đợi 15-30 phút rồi chạy lại cell này.
MIMIC sẵn sàng: 89,976 bản ghi.


## Bước 3 — Nhãn PTB-XL: đọc thật từ `ptbxl_database.csv` + `scp_statements.csv`

Ánh xạ `scp_codes` → `diagnostic_class` (cột thật trong `scp_statements.csv`, lọc `diagnostic==1`)
→ gộp về 3 nhóm `NORM`/`MI`/`OTHER_ABNORMAL`. MIMIC không cần bước này (self-supervised).

In [ ]:
UNIFIED_LABELS = ["NORM", "MI", "OTHER_ABNORMAL"]

def load_ptbxl_database():
    db = pd.read_csv(PTBXL_SOURCE_DIR / "ptbxl_database.csv")
    scp = pd.read_csv(PTBXL_SOURCE_DIR / "scp_statements.csv", index_col=0)
    scp_diag = scp[scp["diagnostic"] == 1.0]

    db["scp_codes"] = db["scp_codes"].apply(ast.literal_eval)

    def _superclasses(codes):
        return sorted({scp_diag.loc[c, "diagnostic_class"] for c in codes if c in scp_diag.index})

    def _unified(classes):
        if classes == ["NORM"]:
            return "NORM"
        if "MI" in classes:
            return "MI"
        return "OTHER_ABNORMAL"

    db["diagnostic_superclass"] = db["scp_codes"].apply(_superclasses)
    db["unified_label"] = db["diagnostic_superclass"].apply(_unified)
    return db

ptbxl_unified = load_ptbxl_database()
if RUN_MODE == "debug":
    ptbxl_unified = ptbxl_unified.sample(n=min(DEBUG_LIMIT, len(ptbxl_unified)),
                                          random_state=SEED).reset_index(drop=True)
else:
    ptbxl_unified = ptbxl_unified.reset_index(drop=True)
ptbxl_unified["_cache_idx"] = np.arange(len(ptbxl_unified))  # giữ vị trí gốc để tra cache sau khi split

print(f"PTB-XL: {len(ptbxl_unified):,} bản ghi | phân bố nhãn: "
      f"{ptbxl_unified['unified_label'].value_counts().to_dict()}")
ptbxl_unified.drop(columns=["scp_codes"]).to_csv(PTBXL_REF_DIR / "ptbxl_labels_manifest.csv", index=False)
# (bỏ cột scp_codes -- kiểu dict lồng nhau ghi CSV không đẹp; diagnostic_superclass/unified_label
# là đủ thông tin cho việc tham khảo/audit lại sau này.)

PTB-XL: 21,799 bản ghi | phân bố nhãn: {'NORM': 9069, 'OTHER_ABNORMAL': 7261, 'MI': 5469}


## Bước 4 — Tiền xử lý (khớp đúng pipeline STEMI) & cache memmap resumable

Highpass 0,05Hz + bandpass 40Hz bậc 3 (zero-phase, `filtfilt`) + winsorize ±6mV — đúng
`stemi_pipeline_optimized_v3.ipynb` mục 6. Cache là **1 file `.npy` lớn mỗi nguồn** (không phải
hàng trăm nghìn file nhỏ), build local trước, backup định kỳ + cuối cùng lên Drive, tự resume nếu
phiên Colab bị ngắt giữa chừng — cùng nguyên tắc mục 6 của pipeline STEMI.

In [ ]:
from scipy.signal import butter, filtfilt, resample as sp_resample

_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")

def resample_to_fs(x, orig_fs, target_fs=FS):
    if orig_fs == target_fs:
        return x
    n_target = int(round(x.shape[-1] * target_fs / orig_fs))
    return sp_resample(x, n_target, axis=-1)

def fit_signal_len(x, target_len=SIGNAL_LEN):
    n = x.shape[-1]
    if n == target_len:
        return x
    if n > target_len:
        return x[:, :target_len]
    return np.pad(x, ((0, 0), (0, target_len - n)))

def preprocess_record(x, orig_fs):
    """x: (NUM_LEADS, n_samples). Khớp đúng preprocess() của stemi_pipeline_optimized_v3.ipynb."""
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = resample_to_fs(x, orig_fs)
    x = fit_signal_len(x)
    x = filtfilt(_B, _A, x, axis=1)
    x = np.clip(x, -WINSORIZE_MV, WINSORIZE_MV)
    return np.ascontiguousarray(x, dtype=np.float32)

def load_raw_signal_ptbxl(row):
    rec = wfdb.rdrecord(str(PTBXL_SOURCE_DIR / row["filename_hr"]))
    return np.asarray(rec.p_signal, dtype=np.float32).T, rec.fs

def load_raw_signal_mimic(rel_path):
    rec = wfdb.rdrecord(str(MIMIC_LOCAL_WORK_DIR / Path(rel_path).name))
    return np.asarray(rec.p_signal, dtype=np.float32).T, rec.fs

def load_and_preprocess_ptbxl(row):
    x, fs = load_raw_signal_ptbxl(row)
    return preprocess_record(x, fs)

def load_and_preprocess_mimic(rel_path):
    x, fs = load_raw_signal_mimic(rel_path)
    return preprocess_record(x, fs)

def _cache_complete(meta_p, npy_p, n_expected):
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= n_expected
    except (OSError, ValueError):
        return False

def build_signal_cache(records, load_fn, cache_tag, checkpoint_every=2000, max_workers=8):
    """records: list phần tử truyền thẳng vào load_fn (row PTB-XL hoặc rel_path MIMIC). Trả về
    memmap (n, NUM_LEADS, SIGNAL_LEN) float16, đúng thứ tự `records`. Resumable qua local + Drive,
    cùng nguyên tắc mục 6 stemi_pipeline_optimized_v3.ipynb (build_cache), có thêm backup định kỳ.

    Đọc + tiền xử lý SONG SONG (max_workers luồng cùng lúc) -- quan trọng vì đọc PTB-XL/MIMIC qua
    Drive FUSE mount có độ trễ mạng mỗi file, tuần tự từng file một rất chậm. Dùng cửa sổ trượt:
    nộp trước max_workers future, TIÊU THỤ kết quả theo đúng thứ tự index để giữ bất biến
    "n_done=K nghĩa là 0..K-1 CHẮC CHẮN đã ghi xong" (an toàn cho resume) trong khi vẫn đọc song
    song nhiều file cùng lúc."""
    n = len(records)
    tag = f"{cache_tag}_{PREPROCESS_VERSION}_n{n}"
    npy_p, meta_p = LOCAL_CACHE_DIR / f"{tag}.npy", LOCAL_CACHE_DIR / f"{tag}.meta.json"
    d_npy_p, d_meta_p = UNIFIED_CACHE_DIR / f"{tag}.npy", UNIFIED_CACHE_DIR / f"{tag}.meta.json"

    if not _cache_complete(meta_p, npy_p, n) and _cache_complete(d_meta_p, d_npy_p, n):
        print(f"[{cache_tag}] Cache đầy đủ trên Drive -- copy về local...")
        shutil.copy2(d_npy_p, npy_p)
        shutil.copy2(d_meta_p, meta_p)

    if _cache_complete(meta_p, npy_p, n):
        print(f"[{cache_tag}] Cache đã đầy đủ ({n:,} bản ghi), dùng lại.")
        return np.load(npy_p, mmap_mode="r")

    meta = json.loads(meta_p.read_text()) if meta_p.exists() else {}
    start = int(meta.get("n_done", 0)) if npy_p.exists() else 0
    if start:
        arr = np.lib.format.open_memmap(npy_p, mode="r+")
        print(f"[{cache_tag}] Build tiếp từ {start:,}/{n:,} (song song {max_workers} luồng)")
    else:
        arr = np.lib.format.open_memmap(npy_p, mode="w+", dtype=np.float16,
                                        shape=(n, NUM_LEADS, SIGNAL_LEN))
        print(f"[{cache_tag}] Build cache mới {n:,} bản ghi "
              f"(~{n * NUM_LEADS * SIGNAL_LEN * 2 / 1024**3:.2f} GB, song song {max_workers} luồng)")

    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        pending = {}
        next_submit = start
        for i in range(start, min(start + max_workers, n)):
            pending[i] = ex.submit(load_fn, records[i])
            next_submit = i + 1
        for i in range(start, n):
            arr[i] = pending.pop(i).result().astype(np.float16)
            if next_submit < n:
                pending[next_submit] = ex.submit(load_fn, records[next_submit])
                next_submit += 1
            done = i + 1
            if done % 200 == 0 or done == n:
                el = max(time.time() - t0, 1e-6)
                print(f"  [{cache_tag}] {done:>7,}/{n:,}  {(done - start) / el:5.1f} rec/s")
            if done % checkpoint_every == 0 or done == n:
                arr.flush()
                meta_p.write_text(json.dumps({"n_done": done}))
                shutil.copy2(npy_p, d_npy_p)
                shutil.copy2(meta_p, d_meta_p)
    del arr
    print(f"[{cache_tag}] Xong trong {time.time() - t0:.0f}s, đã lưu Drive: {d_npy_p}")
    return np.load(npy_p, mmap_mode="r")


ptbxl_cache = build_signal_cache(
    [row for _, row in ptbxl_unified.iterrows()], load_and_preprocess_ptbxl, "ptbxl")
mimic_cache = build_signal_cache(
    mimic_sample_df["_rel_path"].tolist(), load_and_preprocess_mimic, "mimic")
print("PTB-XL cache:", ptbxl_cache.shape, "| MIMIC cache:", mimic_cache.shape)

[ptbxl] Build cache mới 21,799 bản ghi (~2.44 GB)
  [ptbxl]     200/21,799    1.1 rec/s
  [ptbxl]     400/21,799    1.1 rec/s
  [ptbxl]     600/21,799    1.1 rec/s
  [ptbxl]     800/21,799    1.1 rec/s
  [ptbxl]   1,000/21,799    1.1 rec/s
  [ptbxl]   1,200/21,799    1.1 rec/s
  [ptbxl]   1,400/21,799    1.1 rec/s
  [ptbxl]   1,600/21,799    1.2 rec/s
  [ptbxl]   1,800/21,799    1.2 rec/s
  [ptbxl]   2,000/21,799    1.2 rec/s
  [ptbxl]   2,200/21,799    1.1 rec/s
  [ptbxl]   2,400/21,799    1.1 rec/s
  [ptbxl]   2,600/21,799    1.1 rec/s
  [ptbxl]   2,800/21,799    1.1 rec/s
  [ptbxl]   3,000/21,799    1.1 rec/s
  [ptbxl]   3,200/21,799    1.1 rec/s
  [ptbxl]   3,400/21,799    1.1 rec/s
  [ptbxl]   3,600/21,799    1.1 rec/s
  [ptbxl]   3,800/21,799    1.1 rec/s
  [ptbxl]   4,000/21,799    1.1 rec/s
  [ptbxl]   4,200/21,799    1.0 rec/s
  [ptbxl]   4,400/21,799    1.0 rec/s
  [ptbxl]   4,600/21,799    1.0 rec/s
  [ptbxl]   4,800/21,799    1.0 rec/s
  [ptbxl]   5,000/21,799    1.0 rec/s


## Bước 5 — Dataset & DataLoader: PTB-XL (có nhãn, patient-level split) + MIMIC (augment cặp)

`PTBXLLabeledDataset`/`MIMICUnlabeledDataset` đọc trực tiếp từ mảng memmap ở Bước 4 qua cột
`_cache_idx` (giữ đúng vị trí gốc kể cả sau khi patient-level split reset index). Augmentation
(nhiễu Gaussian nhẹ, drop 1–2 chuyển đạo, dịch thời gian) chỉ áp cho MIMIC — tạo cặp dương cho
contrastive loss kiểu SimCLR.

In [ ]:
class PTBXLLabeledDataset(Dataset):
    def __init__(self, cache_array, manifest_df):
        self.cache = cache_array
        self.df = manifest_df.reset_index(drop=True)
        self.label_to_idx = {l: i for i, l in enumerate(UNIFIED_LABELS)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.asarray(self.cache[row["_cache_idx"]], dtype=np.float32)
        y = np.zeros(len(UNIFIED_LABELS), dtype=np.float32)
        y[self.label_to_idx[row["unified_label"]]] = 1.0
        return torch.from_numpy(x), torch.from_numpy(y)

def augment_ecg(x):
    x = x.clone()
    if random.random() < 0.5:
        x = x + torch.randn_like(x) * 0.01
    if random.random() < 0.3:
        n_drop = random.randint(1, 2)
        leads = random.sample(range(x.shape[0]), n_drop)
        x[leads, :] = 0
    if random.random() < 0.5:
        shift = random.randint(-100, 100)
        x = torch.roll(x, shifts=shift, dims=-1)
    return x

class MIMICUnlabeledDataset(Dataset):
    def __init__(self, cache_array, manifest_df):
        self.cache = cache_array
        self.df = manifest_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.from_numpy(np.asarray(self.cache[row["_cache_idx"]], dtype=np.float32))
        return augment_ecg(x), augment_ecg(x)

def patient_level_split_ptbxl(manifest_df, val_frac=0.1, seed=SEED):
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    groups = manifest_df["patient_id"].astype(str)
    train_idx, val_idx = next(gss.split(manifest_df, groups=groups))
    return (manifest_df.iloc[train_idx].reset_index(drop=True),
            manifest_df.iloc[val_idx].reset_index(drop=True))

ptbxl_train_df, ptbxl_val_df = patient_level_split_ptbxl(ptbxl_unified)
assert not (set(ptbxl_train_df["patient_id"]) & set(ptbxl_val_df["patient_id"])), \
    "Rò rỉ bệnh nhân giữa train/val!"

mimic_sample_df["_cache_idx"] = np.arange(len(mimic_sample_df))

BATCH_SIZE = 8 if RUN_MODE == "debug" else 64
ptbxl_train_loader = DataLoader(PTBXLLabeledDataset(ptbxl_cache, ptbxl_train_df),
                                 batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
ptbxl_val_loader = DataLoader(PTBXLLabeledDataset(ptbxl_cache, ptbxl_val_df),
                               batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
mimic_loader = DataLoader(MIMICUnlabeledDataset(mimic_cache, mimic_sample_df),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

# drop_last=True có thể loại bỏ TOÀN BỘ batch nếu dataset nhỏ hơn batch_size (dễ gặp khi
# DEBUG_LIMIT quá nhỏ) -- khi đó vòng train chạy 0 bước và lặng lẽ báo train_loss=0.0000 trông như
# "hoàn hảo" trong khi thực ra không học gì. Chặn tường minh thay vì để lỗi âm thầm.
assert len(ptbxl_train_loader) > 0, (
    f"ptbxl_train_loader rỗng (train={len(ptbxl_train_df)} record, batch_size={BATCH_SIZE}, "
    f"drop_last=True) -- tăng DEBUG_LIMIT hoặc giảm BATCH_SIZE."
)
assert len(mimic_loader) > 0, (
    f"mimic_loader rỗng (mimic={len(mimic_sample_df)} record, batch_size={BATCH_SIZE}, "
    f"drop_last=True) -- tăng DEBUG_LIMIT hoặc giảm BATCH_SIZE."
)
print(f"DataLoader sẵn sàng -- PTB-XL train={len(ptbxl_train_df)} ({len(ptbxl_train_loader)} batch), "
      f"val={len(ptbxl_val_df)} | MIMIC={len(mimic_sample_df)} ({len(mimic_loader)} batch) "
      f"| batch_size={BATCH_SIZE}")

## Bước 6 — Backbone: ResNet1D thật từ `stemi_pipeline_optimized_v3.ipynb` (mục 11)

Lấy nguyên `ResidualBlock1D` + stem + residual blocks của kiến trúc ResNet1D đang dùng thật trong
pipeline STEMI, chỉ bỏ head phân loại STEMI cuối. `cls_head` (BCE, PTB-XL) và `proj_head` (NT-Xent,
MIMIC) là 2 head tạm cho pretrain — bỏ đi sau khi xuất backbone ở Bước 8.

In [ ]:
class ResidualBlock1D(nn.Module):
    """Verbatim từ stemi_pipeline_optimized_v3.ipynb mục 11."""
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)

class ResNet1DBackbone(nn.Module):
    """Stem + residual blocks của ResNet1D thật (stemi_pipeline_optimized_v3.ipynb mục 11),
    KHÔNG PHẢI placeholder -- bỏ head phân loại STEMI cuối để dùng chung cho pretrain."""
    def __init__(self, channels=(32, 64, 128, 256), in_ch=NUM_LEADS):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = c_in

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.pool(x).squeeze(-1)

class ClassificationHead(nn.Module):
    def __init__(self, in_dim, n_labels):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_labels)
    def forward(self, x):
        return self.fc(x)

class ProjectionHead(nn.Module):
    """Head tạm cho contrastive loss -- bỏ đi sau khi pretrain, chỉ giữ backbone."""
    def __init__(self, in_dim, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, in_dim), nn.ReLU(inplace=True), nn.Linear(in_dim, proj_dim))
    def forward(self, x):
        return self.net(x)

class PretrainModel(nn.Module):
    def __init__(self, n_labels=len(UNIFIED_LABELS), proj_dim=128):
        super().__init__()
        self.backbone = ResNet1DBackbone()
        self.cls_head = ClassificationHead(self.backbone.out_dim, n_labels)
        self.proj_head = ProjectionHead(self.backbone.out_dim, proj_dim)
    def forward_cls(self, x):
        return self.cls_head(self.backbone(x))
    def forward_proj(self, x):
        return self.proj_head(self.backbone(x))

def nt_xent_loss(z1, z2, temperature=0.1):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)
    targets = torch.arange(batch_size, device=z.device)
    targets = torch.cat([targets + batch_size, targets], dim=0)
    return F.cross_entropy(sim, targets)

_sanity = PretrainModel()
_x1 = torch.randn(2, NUM_LEADS, SIGNAL_LEN)
_x2 = torch.randn(2, NUM_LEADS, SIGNAL_LEN)
assert _sanity.forward_cls(_x1).shape == (2, len(UNIFIED_LABELS))
assert _sanity.forward_proj(_x2).shape == (2, 128)
n_params = sum(p.numel() for p in _sanity.backbone.parameters())
print(f"Backbone ResNet1D thật -- {n_params:,} tham số. Sanity check OK.")
del _sanity

## Bước 7 — Training loop kết hợp: BCE (PTB-XL) + NT-Xent (MIMIC), resume-from-checkpoint

Chạy thật (không còn comment) khi tới đây — `n_epochs` tự giảm còn 2 ở `RUN_MODE=debug`.

In [ ]:
def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({
        "epoch": epoch, "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(), "val_loss": val_loss,
        "config": {"SEED": SEED, "FS": FS, "SIGNAL_LEN": SIGNAL_LEN, "NUM_LEADS": NUM_LEADS,
                   "PREPROCESS_VERSION": PREPROCESS_VERSION},
    }, path)

def load_checkpoint_if_exists(model, optimizer, path):
    if path.exists():
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        print(f"Resume từ checkpoint epoch={ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")
        return ckpt["epoch"] + 1
    return 0

LAMBDA_SSL = 1.0  # trọng số loss contrastive MIMIC so với loss phân loại PTB-XL

def run_pretrain(ptbxl_train_loader, ptbxl_val_loader, mimic_loader, run_tag="resnet1d_real_pretrain"):
    model = PretrainModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss()

    ckpt_path = MODELS_DIR / f"{run_tag}_latest.pt"
    best_path = MODELS_DIR / f"{run_tag}_best.pt"
    start_epoch = load_checkpoint_if_exists(model, optimizer, ckpt_path)

    n_epochs = 2 if RUN_MODE == "debug" else 40
    best_val_loss = float("inf")

    for epoch in range(start_epoch, n_epochs):
        model.train()
        mimic_iter = iter(mimic_loader)
        train_loss_sum, n_steps = 0.0, 0
        for x_ptbxl, y_ptbxl in ptbxl_train_loader:
            try:
                x_mimic_a, x_mimic_b = next(mimic_iter)
            except StopIteration:
                mimic_iter = iter(mimic_loader)
                x_mimic_a, x_mimic_b = next(mimic_iter)

            x_ptbxl, y_ptbxl = x_ptbxl.to(DEVICE), y_ptbxl.to(DEVICE)
            x_mimic_a, x_mimic_b = x_mimic_a.to(DEVICE), x_mimic_b.to(DEVICE)

            optimizer.zero_grad()
            loss_cls = bce(model.forward_cls(x_ptbxl), y_ptbxl)
            loss_ssl = nt_xent_loss(model.forward_proj(x_mimic_a), model.forward_proj(x_mimic_b))
            loss = loss_cls + LAMBDA_SSL * loss_ssl
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
            n_steps += 1
        train_loss = train_loss_sum / max(n_steps, 1)

        model.eval()
        val_loss_sum, n_val = 0.0, 0
        with torch.no_grad():
            for x, y in ptbxl_val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                val_loss_sum += bce(model.forward_cls(x), y).item()
                n_val += 1
        val_loss = val_loss_sum / max(n_val, 1)

        print(f"[{run_tag}] epoch {epoch + 1}/{n_epochs} -- train_loss={train_loss:.4f} "
              f"val_loss(PTB-XL cls)={val_loss:.4f}")
        save_checkpoint(model, optimizer, epoch, val_loss, ckpt_path)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, epoch, val_loss, best_path)

    return model, best_path

model, best_ckpt_path = run_pretrain(ptbxl_train_loader, ptbxl_val_loader, mimic_loader)

## Bước 8 — Xuất backbone checkpoint (bỏ cả 2 head) + manifest cho TIP-015/016

In [ ]:
def export_backbone_only(best_ckpt_path, run_tag="resnet1d_real_pretrain"):
    ckpt = torch.load(best_ckpt_path, map_location="cpu")
    model = PretrainModel()
    model.load_state_dict(ckpt["model_state"])

    backbone_path = MODELS_DIR / f"{run_tag}_backbone_only.pt"
    torch.save(model.backbone.state_dict(), backbone_path)

    checksum = hashlib.sha256(backbone_path.read_bytes()).hexdigest()
    manifest = {
        "run_tag": run_tag,
        "created_at": datetime.utcnow().isoformat(),
        "run_mode": RUN_MODE,
        "source_best_checkpoint": str(best_ckpt_path),
        "backbone_checkpoint": str(backbone_path),
        "sha256": checksum,
        "backbone_architecture": "ResNet1D (verbatim tu stemi_pipeline_optimized_v3.ipynb muc 11)",
        "pretrain_design": "PTB-XL supervised + MIMIC-IV-ECG self-supervised (NT-Xent), khong can credential",
        "n_ptbxl": len(ptbxl_unified), "n_mimic": len(mimic_sample_df),
        "config": {
            "SEED": SEED, "FS": FS, "SIGNAL_LEN": SIGNAL_LEN, "NUM_LEADS": NUM_LEADS,
            "BP_LOW": BP_LOW, "BP_HIGH": BP_HIGH, "BP_ORDER": BP_ORDER, "WINSORIZE_MV": WINSORIZE_MV,
            "PREPROCESS_VERSION": PREPROCESS_VERSION, "unified_labels": UNIFIED_LABELS,
        },
    }
    manifest_path = MANIFESTS_DIR / f"{run_tag}_run_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print(f"Backbone checkpoint: {backbone_path}\nManifest: {manifest_path}\nSHA-256: {checksum}")
    return backbone_path, manifest_path

backbone_path, manifest_path = export_backbone_only(best_ckpt_path)

## Kết thúc — bàn giao cho TIP-015 / TIP-016

`manifests/*_run_manifest.json` ghi lại checksum + toàn bộ config đã dùng. Hai notebook fine-tune
(STEMI, OMI) load `*_backbone_only.pt` để khởi tạo thay vì random init, rồi chạy lại đúng bộ 6 mức
freeze/unfreeze như đã làm với ECGFounder ở Phần I trước khi so sánh với baseline from-scratch hiện
có trong `Bao_cao_nghien_cuu.docx`.

**Vị trí Drive:** notebook + toàn bộ dữ liệu/checkpoint pretrain nằm trong
`ACS-ECG-AI_pretrain_finetune/` ở MyDrive riêng — tách biệt với `ACS-ECG-AI/` (được share).

**Không cần credential PhysioNet** — chỉ dùng waveform mở của MIMIC-IV-ECG và PTB-XL. Nếu
`record_list.csv` (Bước 2) hoá ra cũng bị gate, đó là file duy nhất cần nhờ người có credential tải
hộ — không cần Ext-ICD.

**Giả định còn lại, kiểm tra nếu full run thất bại ở bước tương ứng:**
- Bước 3/4: chuẩn PTB-XL 1.0.3 gốc (cột `filename_hr`, `scp_codes`, `patient_id` trong
  `ptbxl_database.csv`; cột `diagnostic_class`/`diagnostic` trong `scp_statements.csv`) — đã xác
  nhận khớp với dữ liệu thật trong `VinAMI_ACS/data/ptb-xl/1.0.3/`.
- Bước 2: tên cột path trong `record_list.csv` của MIMIC-IV-ECG được dò tự động
  (`find_path_column`) thay vì giả định cứng — nếu vẫn sai, in ra để kiểm tra thủ công.
- Bước 6: kiến trúc backbone là **ResNet1D thật**, không còn placeholder. Muốn thử kiến trúc khác
  trong 10 kiến trúc của `stemi_pipeline_optimized_v3.ipynb` (XResNet1D, ConvNeXtV2_1D...) thì copy
  class tương ứng từ mục 11 của notebook đó vào thay `ResNet1DBackbone`.